<a href="https://colab.research.google.com/github/phornkanok00/GE338-Data-Science/blob/main/Lab5_Global_cooling/Lab5_Feasibility_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การวิเคราะห์พื้นที่การอยู่รอดของมนุษย์ภายใต้สถานการณ์อุณหภูมิลดลง
* This notebook analyzes the impact of a -5°C global temperature scenario on Thailand using NDVI, rainfall, and temperature data.

In [ ]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.2 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-phornkanok')

In [ ]:
import geemap
Map = geemap.Map(center=[15, 100], zoom=6)

# ===========================
# 1. Thailand Boundary
# ===========================
thailand = ee.FeatureCollection("FAO/GAUL/2015/level0") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

# จังหวัด (level1)
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

Map.addLayer(provinces, {}, 'Provinces')

# ===========================
# 2. Load Data (2020–2023)
# ===========================

# NDVI
ndvi = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate('2020-01-01', '2023-12-31') \
    .select('NDVI') \
    .mean() \
    .multiply(0.0001) \
    .clip(thailand)

# Temperature
temp = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
    .filterDate('2020-01-01', '2023-12-31') \
    .select('temperature_2m') \
    .mean() \
    .subtract(273.15) \
    .clip(thailand)

# Rainfall
rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
    .filterDate('2020-01-01', '2023-12-31') \
    .select('precipitation') \
    .mean() \
    .clip(thailand)

# ===========================
# 3. Scenario: Temp -5°C
# ===========================
temp_scenario = temp.subtract(5)

# ===========================
# 4. Normalize
# ===========================
ndvi_norm = ndvi.unitScale(0, 1)
rain_norm = rain.unitScale(0, 300)
temp_norm = temp_scenario.unitScale(-20, 40)

# ===========================
# 5. FAI
# ===========================
fai = ndvi_norm.multiply(0.5).add(rain_norm.multiply(0.5))

# ===========================
# 6. GSI
# ===========================
gsi = fai.multiply(0.4).add(temp_norm.multiply(0.6))

# ===========================
# 7. Classification
# ===========================
survival = gsi.gt(0.55)
risk = gsi.gt(0.45).And(gsi.lte(0.55))
collapse = gsi.lte(0.45)


In [ ]:
Map = geemap.Map(center=[15, 100], zoom=6)

# GSI
Map.addLayer(gsi, {
    'min': 0,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}, 'GSI')

# Zones (รวมเป็น layer เดียว จะเห็นชัดกว่า)
zone = ee.Image(0) \
    .where(survival, 3) \
    .where(risk, 2) \
    .where(collapse, 1)

zone_vis = zone.updateMask(zone.neq(0))

Map.addLayer(zone_vis, {
    'min': 1,
    'max': 3,
    'palette': ['red', 'yellow', 'green']
}, 'Zones')


Map.addLayer(thailand, {}, 'Thailand boundary')

Map

Map(center=[15, 100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

In [ ]:
pixelArea = ee.Image.pixelArea().divide(1e6)

areaSurvival = pixelArea.updateMask(survival).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=thailand,
    scale=5000,
    maxPixels=1e13
)

areaRisk = pixelArea.updateMask(risk).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=thailand,
    scale=5000,
    maxPixels=1e13
)

areaCollapse = pixelArea.updateMask(collapse).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=thailand,
    scale=5000,
    maxPixels=1e13
)

print("Survival:", areaSurvival.getInfo())
print("Risk:", areaRisk.getInfo())
print("Collapse:", areaCollapse.getInfo())

Survival: {'area': 105628.47318297355}
Risk: {'area': 402678.1592072214}
Collapse: {'area': 315.5426437411764}


In [ ]:
hist = gsi.reduceRegion(
    reducer=ee.Reducer.histogram(),
    geometry=thailand,
    scale=5000,
    maxPixels=1e13
)

print(hist.getInfo())

{'NDVI': {'bucketMeans': [0.3964676704669728, 0.39697265625, 0.39794921875, 0.39892578125, 0.39990234375, 0.40087890625, 0.40185546875, 0.40283203125, 0.40380859375, 0.40478515625, 0.40576171875, 0.40673828125, 0.40771484375, 0.40869140625, 0.40966796875, 0.41064453125, 0.41162109375, 0.41259765625, 0.41357421875, 0.41455078125, 0.41552734375, 0.41650390625, 0.41770332630114515, 0.4183371816690878, 0.41943359375, 0.42041015625, 0.42138671875, 0.42236328125, 0.4234058277823335, 0.42431640625, 0.42529296875, 0.42627823414780497, 0.42724609375, 0.42822265625, 0.42935286605979095, 0.43050097854981345, 0.4315579811024926, 0.4317876112638583, 0.43338905180177906, 0.43408203125, 0.4347498230578765, 0.43603515625, 0.43709594013861125, 0.43798828125, 0.43896484375, 0.43994140625, 0.44091796875, 0.44189453125, 0.44287109375, 0.4438427858397863, 0.44482421875, 0.44570478627905336, 0.4466345181399142, 0.4476672082879153, 0.44873046875, 0.44938767154950054, 0.45104366492761805, 0.45151160350124736,

# ทดสอบทำโมเดล scenario in future

In [ ]:
# ===========================
# 1. THAILAND
# ===========================
thailand = ee.FeatureCollection("FAO/GAUL/2015/level0") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

# ===========================
# 2. LOAD DATA (2020–2023)
# ===========================
ndvi = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate('2020-01-01', '2023-12-31') \
    .select('NDVI') \
    .mean() \
    .multiply(0.0001) \
    .clip(thailand)

temp = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
    .filterDate('2020-01-01', '2023-12-31') \
    .select('temperature_2m') \
    .mean() \
    .subtract(273.15) \
    .clip(thailand)

rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
    .filterDate('2020-01-01', '2023-12-31') \
    .select('precipitation') \
    .mean() \
    .clip(thailand)

# ===========================
# 3. NORMALIZE
# ===========================
ndvi_norm = ndvi.unitScale(0, 1)
rain_norm = rain.unitScale(0, 300)
temp_norm = temp.unitScale(-20, 40)

# ===========================
# 4. FAI + GSI (ปรับ weight แล้ว)
# ===========================
fai = ndvi_norm.multiply(0.5).add(rain_norm.multiply(0.5))

# ปรับให้ realistic ขึ้น
gsi = fai.multiply(0.7).add(temp_norm.multiply(0.3))

# ===========================
# 5. CLASS จาก GSI (Current)
# ===========================
label_current = gsi.expression(
    "(gsi <= 0.35) ? 1" +
    ": (gsi <= 0.5) ? 2" +
    ": 3",
    {"gsi": gsi}
)

# ===========================
# 6. FEATURES
# ===========================
features = ndvi_norm.rename('ndvi') \
    .addBands(rain_norm.rename('rain')) \
    .addBands(temp_norm.rename('temp'))

dataset = features.addBands(label_current.rename('label'))

# ===========================
# 7. SAMPLE
# ===========================
samples = dataset.sample(
    region=thailand,
    scale=5000,
    numPixels=5000,
    seed=42
)

# ===========================
# 8. TRAIN RF
# ===========================
rf = ee.Classifier.smileRandomForest(50).train(
    features=samples,
    classProperty='label',
    inputProperties=['ndvi', 'rain', 'temp']
)

# ===========================
# 9. FUTURE SCENARIOS
# ===========================
temp_2030 = temp.add(2).unitScale(-20, 40)
temp_2040 = temp.add(4).unitScale(-20, 40)

features_2030 = ndvi_norm.rename('ndvi') \
    .addBands(rain_norm.rename('rain')) \
    .addBands(temp_2030.rename('temp'))

features_2040 = ndvi_norm.rename('ndvi') \
    .addBands(rain_norm.rename('rain')) \
    .addBands(temp_2040.rename('temp'))

# RF prediction
pred_2030 = features_2030.classify(rf)
pred_2040 = features_2040.classify(rf)

# ===========================
# 10. MAP
# ===========================
Map = geemap.Map(center=[15, 100], zoom=6)

# current = GSI จริง (ไม่ใช้ RF)
current_vis = label_current.toByte().reproject(crs='EPSG:4326', scale=10000)

future_2030_vis = pred_2030.toByte().reproject(crs='EPSG:4326', scale=10000)
future_2040_vis = pred_2040.toByte().reproject(crs='EPSG:4326', scale=10000)

Map.addLayer(current_vis, {
    'min': 1, 'max': 3,
    'palette': ['red', 'yellow', 'green']
}, 'Current (GSI)')

Map.addLayer(future_2030_vis, {
    'min': 1, 'max': 3,
    'palette': ['red', 'yellow', 'green']
}, '2030 (RF)')

Map.addLayer(future_2040_vis, {
    'min': 1, 'max': 3,
    'palette': ['red', 'yellow', 'green']
}, '2040 (RF)')

Map.addLayer(thailand, {}, 'Thailand')

Map

Map(center=[15, 100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

# Feasibility และ Risk Assessment

1.   ความเสี่ยงหลักโครงการ
* ค่า GSI กระจุกทำให้แผนที่แสดงออกมาเป็นสีเดียว หรือเดียวอย่างค่า risk หาบริเวณอยู่รอดไม่เจอ
* ความละเอียดข้อมูลไม่เท่ากัน ในแต่ละปัจจัย
2.   Plan B ถ้าเจอปัญหาแต่ละอย่าง
* ถ้าค่าไม่กระจายให้ลองปรับ threshold หรือ ใช้ percentile classification
* กรณีข้อมูลไม่เท่ากันให้ Resample ข้อมูลเป็นรายละเอียดเดียวกัน
3. ขอบเขตที่จะ Descope ออกถ้าเวลาไม่พอ
* การวิเคราะห์หลาย scenario
* การปรับ parameter หลายรอบ
* การทำ comparison หลาย dataset


